# 02 - Signal Gate: Suraj520 vs CFPB

**Support Ticket Triage** - Notebook 2 of 6

Notebook 01 found that the Suraj520 support-ticket dataset has **identical top TF-IDF
terms across every category and every priority level**, Cramer's V near zero on all
field pairs, and suspiciously uniform label distributions (priority imbalance ratio
1.06). The conclusion was that the labels are independent of the ticket text.

This notebook turns that qualitative read into **one citable number**, then applies the
*same test with the same code* to the replacement dataset.

The gate: **does TF-IDF + Logistic Regression beat a stratified dummy by >= 0.03
macro-F1?** If not, the labels carry no learnable lexical signal, and a 66M-parameter
transformer will not conjure one.

| | Expectation |
|---|---|
| Suraj520 - Ticket Type / Priority | **fails** - confirms Notebook 01 |
| CFPB - Product / Issue | **passes** - real narratives, real human-assigned labels |

*Accelerator: **None (CPU)**. Internet: ON (or add both datasets via Add Input).*

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
sns.set_theme(style="whitegrid")

SEED = 42
WORK = "/kaggle/working"
PLOTS = os.path.join(WORK, "plots")
os.makedirs(PLOTS, exist_ok=True)

def savefig(name):
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS, name + ".png"), dpi=120, bbox_inches="tight")
    plt.show()

def find_files(base, exts=(".csv", ".tsv")):
    out = []
    if not os.path.isdir(base):
        return out
    for root, _, files in os.walk(base):
        for f in files:
            if f.lower().endswith(exts):
                out.append(os.path.join(root, f))
    return out

def pick(cols, *candidates):
    """First candidate present in cols (case/space-insensitive), else None."""
    norm = {str(c).lower().replace("_", " ").strip(): c for c in cols}
    for cand in candidates:
        k = cand.lower().replace("_", " ").strip()
        if k in norm:
            return norm[k]
    return None

print("ready")

---
## The gate

One function, used identically on both datasets. Two deliberate choices:

- **The split here is throwaway.** It exists only to score the gate. The project's real
  70/15/15 split happens in Notebook 03, and the test set stays sealed until Notebook 06.
- **Both datasets are scored at the same sample size.** Otherwise CFPB would win on
  volume alone and the comparison would prove nothing.

In [ ]:
GATE_MARGIN = 0.03   # required macro-F1 lift over the stratified dummy

def signal_gate(texts, labels, name, n_sample=None, seed=SEED, verbose=True,
                min_per_class=10):
    """Dummy vs TF-IDF+LogReg macro-F1. Returns a result dict."""
    s = pd.DataFrame({"text": texts, "y": labels}).dropna()
    s = s[s["text"].astype(str).str.strip().str.len() > 0]

    # Subsample FIRST, then drop classes too rare to stratify.
    # Filtering before sampling is a bug: a class with 12 rows in a 1M-row frame
    # can land 1 row in a 25k sample, and train_test_split(stratify=...) then
    # raises "The least populated class in y has only 1 member".
    if n_sample and len(s) > n_sample:
        s = s.sample(n_sample, random_state=seed)

    vc = s["y"].value_counts()
    keep = vc[vc >= min_per_class].index
    n_dropped = int((~s["y"].isin(keep)).sum())
    s = s[s["y"].isin(keep)]
    if n_dropped and verbose:
        print(f"  (dropped {n_dropped} rows in classes with < {min_per_class} examples)")

    Xtr, Xva, ytr, yva = train_test_split(
        s["text"].astype(str), s["y"], test_size=0.2,
        stratify=s["y"], random_state=seed,
    )

    dummy = DummyClassifier(strategy="stratified", random_state=seed).fit(Xtr, ytr)
    f1_dummy = f1_score(yva, dummy.predict(Xva), average="macro")

    tfidf = make_pipeline(
        TfidfVectorizer(max_features=50_000, ngram_range=(1, 2),
                        min_df=2, sublinear_tf=True, strip_accents="unicode"),
        LogisticRegression(max_iter=1000, class_weight="balanced", random_state=seed),
    ).fit(Xtr, ytr)
    pred = tfidf.predict(Xva)
    f1_tfidf = f1_score(yva, pred, average="macro")

    lift = f1_tfidf - f1_dummy
    passed = lift >= GATE_MARGIN

    res = {
        "task": name,
        "n_used": int(len(s)),
        "n_classes": int(s["y"].nunique()),
        "f1_dummy": round(float(f1_dummy), 4),
        "f1_tfidf": round(float(f1_tfidf), 4),
        "lift": round(float(lift), 4),
        "accuracy_tfidf": round(float(accuracy_score(yva, pred)), 4),
        "n_dropped_rare": n_dropped,
        "passed": bool(passed),
    }
    if verbose:
        flag = "PASS" if passed else "FAIL"
        print(f"[{flag}] {name}")
        print(f"       n={res['n_used']}  classes={res['n_classes']}")
        print(f"       dummy macro-F1 = {res['f1_dummy']:.4f}")
        print(f"       TF-IDF macro-F1 = {res['f1_tfidf']:.4f}")
        print(f"       lift = {res['lift']:+.4f}   (need >= {GATE_MARGIN})\n")
    return res

results = []

---
# Part 1 - Suraj520 (the incumbent)

Producing the number the README will cite.

In [ ]:
sur_csvs = [p for p in find_files("/kaggle/input") if "customer_support_tickets" in p.lower()]

if not sur_csvs:
    import kagglehub
    base = kagglehub.dataset_download("suraj520/customer-support-ticket-dataset")
    sur_csvs = [p for p in find_files(base) if "customer_support" in p.lower()]

print(sur_csvs)
sur = pd.read_csv(sur_csvs[0])
print("shape:", sur.shape)

MATCH_N = len(sur)          # every gate below is scored at this sample size
print("matched sample size for all gates:", MATCH_N)

In [ ]:
results.append(signal_gate(sur["Ticket Description"], sur["Ticket Type"],
                          "Suraj520 / Ticket Type (5 classes)"))
results.append(signal_gate(sur["Ticket Description"], sur["Ticket Priority"],
                          "Suraj520 / Ticket Priority (4 classes)"))
results.append(signal_gate(sur["Ticket Description"], sur["Ticket Subject"],
                          "Suraj520 / Ticket Subject (16 classes)"))

`Ticket Subject` is included as the last chance for this dataset - it was the one
alternative target Notebook 01 flagged but did not test. If it fails too, no target
derived from this text is learnable, and the swap is fully justified.

---
# Part 2 - CFPB Consumer Complaint Database

Real complaints written by real people, with **human-assigned** `Product` and `Issue`
labels. These become the two classification tasks, replacing category and priority.

**Add the dataset first** via *Add Input -> Datasets*. Several CFPB mirrors exist on
Kaggle with different column naming; the loader below resolves columns by role, so any
of them works. Set `CFPB_SLUG` to whichever you added.

**Memory matters here** - the full database is multi-GB. The loader reads the header
first, resolves the three columns it needs, and only then reads those columns.

In [ ]:
CFPB_SLUG = "selener/consumer-complaint-database"   # change if you added a different mirror

cfpb_files = [p for p in find_files("/kaggle/input")
              if "customer_support_tickets" not in p.lower()]

if not cfpb_files:
    import kagglehub
    base = kagglehub.dataset_download(CFPB_SLUG)
    cfpb_files = find_files(base)

print("candidate files:")
for p in cfpb_files:
    print(" ", p, f"({os.path.getsize(p) / 1e6:.0f} MB)")

CFPB = max(cfpb_files, key=os.path.getsize)   # the complaints table is the big one
print("\nusing:", CFPB)

In [ ]:
# Read the header only, resolve columns by role, then read just those columns.
head = pd.read_csv(CFPB, nrows=5)
print("columns found:", list(head.columns), "\n")

NARR = pick(head.columns, "Consumer complaint narrative", "consumer_complaint_narrative",
            "complaint_what_happened", "narrative")
PROD = pick(head.columns, "Product", "product")
ISSU = pick(head.columns, "Issue", "issue")

print("narrative ->", NARR)
print("product   ->", PROD)
print("issue     ->", ISSU)
assert NARR and PROD and ISSU, "Could not resolve columns - inspect the list above."

cf = pd.read_csv(CFPB, usecols=[NARR, PROD, ISSU], dtype=str, low_memory=False)
print("\nraw rows:", len(cf))

cf = cf.dropna(subset=[NARR, PROD, ISSU])
cf = cf[cf[NARR].str.strip().str.len() > 20]
print("rows with a usable narrative:", len(cf))

### Apply the Notebook 01 checks to the new data

Same scrutiny, before trusting it. The contrast that matters most is
**class balance**: Suraj520's uniformity (imbalance ratio 1.06) was the signature of
random assignment. Real-world label distributions are heavily skewed.

In [ ]:
print("=== narrative length (words) ===")
wl = cf[NARR].str.split().str.len()
print(wl.describe().round(1).to_string())
print(f"p95 = {wl.quantile(0.95):.0f} words")
print(f"rows likely truncated at max_length=128: {(wl > 98).mean():.1%}")
print(f"rows likely truncated at max_length=256: {(wl > 197).mean():.1%}")

print(f"\n=== duplication ===")
dup = 1 - cf[NARR].nunique() / len(cf)
print(f"duplication rate: {dup:.2%}")

for col in (PROD, ISSU):
    vc = cf[col].value_counts()
    print(f"\n=== {col}: {cf[col].nunique()} classes ===")
    print(vc.head(12).to_string())
    print(f"imbalance ratio (max/min) = {vc.max() / vc.min():.1f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
for a, col in zip(ax, (PROD, ISSU)):
    top = cf[col].value_counts().head(15)
    sns.barplot(x=top.values, y=top.index, ax=a)
    a.set_title(f"CFPB {col} - top 15 of {cf[col].nunique()}")
    a.set_xlabel("count")
savefig("04_cfpb_label_distributions")

### Restrict `Issue` to the top classes

`Issue` has a long tail of very rare values, and CFPB's label taxonomy changed over
time. Keeping the top 15 gives a well-posed multi-class problem instead of one
dominated by classes with a handful of examples.

In [ ]:
TOP_ISSUES = 15
top_issues = cf[ISSU].value_counts().head(TOP_ISSUES).index
cf_issue = cf[cf[ISSU].isin(top_issues)]

print(f"Issue restricted to top {TOP_ISSUES} classes: {len(cf_issue)} rows "
      f"({len(cf_issue) / len(cf):.1%} of narratives)")
print(cf[ISSU].value_counts().head(TOP_ISSUES).to_string())

In [ ]:
results.append(signal_gate(cf[NARR], cf[PROD],
                          f"CFPB / Product ({cf[PROD].nunique()} classes)",
                          n_sample=MATCH_N))
results.append(signal_gate(cf_issue[NARR], cf_issue[ISSU],
                          f"CFPB / Issue (top {TOP_ISSUES})",
                          n_sample=MATCH_N))

### Does more data help?

The gate above is scored at Suraj520's sample size so the comparison is fair. This cell
re-runs CFPB at a realistic training size - useful for choosing how much data to
actually fine-tune on in Week 2, since GPU time is the binding constraint.

In [ ]:
for n in (25_000, 100_000):
    if len(cf) >= n:
        r = signal_gate(cf[NARR], cf[PROD], f"CFPB / Product @ n={n:,}", n_sample=n)
        results.append(r)

---
## Verdict

In [ ]:
tbl = pd.DataFrame(results)[
    ["task", "n_used", "n_classes", "f1_dummy", "f1_tfidf", "lift", "passed"]
]
display(tbl)

print("\n" + "=" * 72)
for r in results:
    mark = "PASS" if r["passed"] else "FAIL"
    print(f"{mark:5s} {r['task']:45s} lift = {r['lift']:+.4f}")
print("=" * 72)

gate = {
    "gate_margin": GATE_MARGIN,
    "matched_sample_size": int(MATCH_N),
    "results": results,
    "decision": "Suraj520 rejected (no lexical signal); CFPB adopted."
                " Tasks: Product + Issue classification.",
}
with open(os.path.join(WORK, "gate_results.json"), "w") as f:
    json.dump(gate, f, indent=2)

tbl.to_csv(os.path.join(WORK, "gate_results.csv"), index=False)
print("\nwrote gate_results.json / .csv")

In [ ]:
plot_df = pd.DataFrame(results)
plot_df = plot_df[~plot_df["task"].str.contains("@ n=")]
melt = plot_df.melt(id_vars="task", value_vars=["f1_dummy", "f1_tfidf"],
                    var_name="model", value_name="macro_f1")

plt.figure(figsize=(10, 5))
ax = sns.barplot(data=melt, y="task", x="macro_f1", hue="model")
ax.set(title=f"Signal gate: TF-IDF must beat dummy by >= {GATE_MARGIN} macro-F1",
       xlabel="validation macro-F1", ylabel="")
savefig("05_signal_gate")

---
## Next

If CFPB passed and Suraj520 failed, the swap is evidenced and the project continues on
CFPB with **Product** and **Issue** as the two classification tasks.

Carry forward into Notebook 03:

- `gate_results.json` - the citable numbers for the README
- The narrative-length percentiles above decide `max_length` (128 vs 256) in Week 2.
  CFPB narratives are far longer than Suraj520's 57-word p95, so **check this** rather
  than reusing 128 out of habit.
- `TOP_ISSUES = 15` and the resolved column names

**Notebook 03** does the real 70/15/15 stratified split, seals the test set behind
`load_split()`, and establishes the TF-IDF baselines properly.

**Before leaving:** *Save Version -> Save & Run All*.